# Starlink broadband 真实轨道 CSV

读取 `download_tle.ipynb` 固定的本地快照，使用 Skyfield/SGP4 传播；本 notebook 不联网。沿用参考代码的 `[DTC]` 排除规则及 450 km 门限。 高度筛选只是工程近似，不能单凭 TLE 判断实际业务状态。

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Locate the shared module from either the repository or notebook directory.
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate satellite"]
             if (p / "satellite_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the WorldNTN repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from satellite_pipeline import (
    download_snapshot, initialize_scenario, read_scenario, load_snapshot,
    export_tracks, preview_export, merge_exports, stamp,
)
# If dependencies are missing, run this in a separate notebook cell:
# %pip install -r "{ROOT / 'requirements.txt'}"
print("Workspace:", ROOT)


In [ ]:
CONSTELLATION = "starlink"
scenario, times = read_scenario(ROOT)
print("UTC range:", times[0], "to", times[-1], "samples:", len(times))
print("Minimum orbit altitude (km):", scenario["min_orbit_altitude_km"][CONSTELLATION])
display(scenario)


## 读取真实目录
默认读取完整 OMM JSON。需要旧 TLE 文件时可显式使用 `load_tle_file(path, CONSTELLATION)`，但传统 TLE 可能漏掉六位编号。

In [ ]:
satellites = load_snapshot(CONSTELLATION, root=ROOT)
print("Catalog satellites:", len(satellites))


## 生成 CSV
默认导出全部符合门限的可见卫星，不进行 top-N 截断。保留原 ENU 列并增加 ECEF 位置/速度、经纬度、距离变化率、唯一 ID、时间索引和来源。`scenario.json` 中 `write_all_states=true` 可额外保存全球保留卫星状态（含观测点不可见者），文件可能较大。速度由传播器和旋转参考系转换直接计算。

In [ ]:
result = export_tracks(satellites, CONSTELLATION, root=ROOT)
display(preview_export(result))


In [ ]:
counts = pd.read_csv(result["counts_csv"])
audit = pd.read_csv(result["audit_csv"])
display(counts.describe())
display(audit.head(10))
print("Audit:", result["audit_csv"])


零可见时刻也保存在 `counts.csv` 中并参与平均值计算。过滤原因、过期历元和传播错误计数在 `filter_audit.csv`；完整设置及输入哈希在 `metadata.json`。